# 🩺 Evaluating Vision Transformer Adaptation for X-Ray Disease Classification

**Objective:** Evaluate partial, full, and parameter-efficient fine-tuning (PEFT) approaches on a multi-label chest X-ray dataset, comparing them to convolutional baselines (e.g., DenseNet-121) to quantify domain adaptation impact.

---

## Step 1: Environment Setup & Imports
Ensure you have the required libraries installed:
`pip install torch torchvision transformers peft scikit-learn pandas matplotlib numpy Pillow`

In [1]:
import torch
import numpy as np

print(torch.__version__)
print(torch.cuda.is_available())
print(np.__version__)

2.7.1+cu118
True
2.4.3


In [3]:
import os
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader, Dataset
from transformers import ViTForImageClassification, ViTConfig
from peft import LoraConfig, get_peft_model
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


## Step 2: Data Loading & Preprocessing
Medical images require specific augmentation. Multi-label datasets (like NIH ChestX-ray14) require vectors of floats for `BCEWithLogitsLoss`.

In [ ]:
from PIL import Image
import glob
import numpy as np

DISEASE_LABELS = [
    "Atelectasis", "Cardiomegaly", "Consolidation", "Edema", "Effusion",
    "Emphysema", "Fibrosis", "Hernia", "Infiltration", "Mass",
    "Nodule", "Pneumonia", "Pneumothorax", "Pleural_Thickening"
]
NUM_CLASSES = len(DISEASE_LABELS)

class ChestXRayDataset(Dataset):
    def __init__(self, csv_file: str, image_dirs: list, transform=None, labels_filter: list = None):
        self.df = pd.read_csv(csv_file)
        self.image_dirs = image_dirs
        self.transform = transform
        self.labels_filter = labels_filter or DISEASE_LABELS
        
        self.image_names = self.df['Image Index'].values
        self._build_label_matrix()
        self._build_image_path_cache()
    
    def _build_image_path_cache(self):
        self.path_cache = {}
        for img_dir in self.image_dirs:
            for img_path in glob.glob(os.path.join(img_dir, "*.png")):
                img_name = os.path.basename(img_path)
                self.path_cache[img_name] = img_path
    
    def _find_image_path(self, img_name):
        if img_name in self.path_cache:
            return self.path_cache[img_name]
        for img_dir in self.image_dirs:
            potential_path = os.path.join(img_dir, img_name)
            if os.path.exists(potential_path):
                return potential_path
        return None

    def _build_label_matrix(self) -> None:
        label_vectors = []
        for _, row in self.df.iterrows():
            findings = row['Finding Labels'].split('|')
            if findings == ['No Finding'] or findings[0] == 'No Finding':
                vector = [0.0] * len(self.labels_filter)
            else:
                vector = [1.0 if disease in findings else 0.0 for disease in self.labels_filter]
            label_vectors.append(vector)
        self.labels = np.array(label_vectors, dtype=np.float32)

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int):
        img_name = self.image_names[idx]
        img_path = self._find_image_path(img_name)
        if img_path is None:
            raise FileNotFoundError(f"Image not found: {img_name}")
        image = Image.open(img_path).convert('RGB')
        labels = torch.tensor(self.labels[idx])
        
        if self.transform:
            image = self.transform(image)
            
        return image, labels

    def get_label_distribution(self):
        counts = self.labels.sum(axis=0)
        for disease, count in zip(self.labels_filter, counts):
            pct = count / len(self) * 100
            print(f"{disease}: {int(count):5d} ({pct:.2f}%)")
        print(f"\n'No Finding' (all zeros): {int((self.labels.sum(axis=1) == 0).sum())} samples")


transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

DATA_DIR = "data"
CSV_PATH = os.path.join(DATA_DIR, "Data_Entry_2017.csv")
IMAGE_DIRS = [
    os.path.join(DATA_DIR, "images_001", "images"),
    os.path.join(DATA_DIR, "images_002", "images"),
    os.path.join(DATA_DIR, "images_003", "images"),
    os.path.join(DATA_DIR, "images_004", "images"),
    os.path.join(DATA_DIR, "images_005", "images"),
    os.path.join(DATA_DIR, "images_006", "images"),
    os.path.join(DATA_DIR, "images_007", "images"),
    os.path.join(DATA_DIR, "images_008", "images"),
    os.path.join(DATA_DIR, "images_009", "images"),
    os.path.join(DATA_DIR, "images_010", "images"),
    os.path.join(DATA_DIR, "images_011", "images"),
    os.path.join(DATA_DIR, "images_012", "images"),
]
dataset = ChestXRayDataset(CSV_PATH, IMAGE_DIRS, transform=transform)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

print(f"Dataset: {len(dataset)} samples, {NUM_CLASSES} classes")
print("\nLabel distribution:")
dataset.get_label_distribution()


## Step 3: Baseline Model (CNN)
DenseNet-121 is the gold standard baseline for Chest X-ray classification.

In [ ]:
def build_baseline_cnn(num_classes):
    model = models.densenet121(weights='IMAGENET1K_V1')
    num_ftrs = model.classifier.in_features
    # Replace classifier for multi-label (No Sigmoid here, handled by loss function)
    model.classifier = nn.Linear(num_ftrs, num_classes)
    return model

baseline_model = build_baseline_cnn(num_classes=14).to(device)

## Step 4: Vision Transformer (ViT) Adaptation Strategies
Here we implement the core of the study: Full Fine-tuning, Partial Fine-tuning (linear probing + top layers), and PEFT (LoRA).

In [ ]:
def build_vit_model(strategy="full", num_classes=14):
    # Load pre-trained ViT from Hugging Face
    model = ViTForImageClassification.from_pretrained(
        "google/vit-base-patch16-224-in21k",
        num_labels=num_classes,
        ignore_mismatched_sizes=True
    )

    if strategy == "full":
        # Standard full fine-tuning; all params require grad
        pass 

    elif strategy == "partial":
        # Freeze embeddings and the first 10 transformer blocks
        for param in model.vit.embeddings.parameters():
            param.requires_grad = False
        for layer in model.vit.encoder.layer[:-2]: # Keep last 2 layers trainable
            for param in layer.parameters():
                param.requires_grad = False

    elif strategy == "peft_lora":
        # Inject Low-Rank Adapters (LoRA) into the attention modules
        config = LoraConfig(
            r=16,
            lora_alpha=16,
            target_modules=["query", "value"], # Target self-attention
            lora_dropout=0.1,
            bias="none",
            modules_to_save=["classifier"] # Ensure head is trainable
        )
        model = get_peft_model(model, config)
        model.print_trainable_parameters()

    return model

# Example instantiation:
# vit_lora = build_vit_model(strategy="peft_lora").to(device)

## Step 5: Training Loop
Using `BCEWithLogitsLoss` for multi-label stability.

In [ ]:
def train_one_epoch(model, dataloader, optimizer, criterion):
    model.train()
    running_loss = 0.0
    
    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        
        # HF ViT returns an object containing logits; torchvision CNNs return logits directly
        logits = outputs.logits if hasattr(outputs, 'logits') else outputs
        
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
    return running_loss / len(dataloader)

criterion = nn.BCEWithLogitsLoss()

# Sanity check: train on a single batch for 10 epochs
print("\n=== SANITY CHECK ===")
print(f"Running on {device}")

sanity_dataloader = DataLoader(dataset, batch_size=16, shuffle=True)
sanity_batch = next(iter(sanity_dataloader))
sanity_images, sanity_labels = sanity_batch[0].to(device), sanity_batch[1].to(device)
print(f"Single batch shape: images={sanity_images.shape}, labels={sanity_labels.shape}")

model = build_baseline_cnn(num_classes=14).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

print("\nTraining for 10 epochs on single batch...")
for epoch in range(10):
    model.train()
    optimizer.zero_grad()
    outputs = model(sanity_images)
    loss = criterion(outputs, sanity_labels)
    loss.backward()
    optimizer.step()
    print(f"Epoch {epoch+1}/10 - Loss: {loss.item():.4f}")

print("\nSanity check complete! Model trained without crashing.")
print("====================\n")

## Step 6: Evaluation & Metrics
Multi-label tasks generally rely on Mean AUC-ROC across all classes.

In [ ]:
def evaluate(model, dataloader):
    model.eval()
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            outputs = model(images)
            logits = outputs.logits if hasattr(outputs, 'logits') else outputs
            
            # Apply sigmoid to convert logits to probabilities
            probs = torch.sigmoid(logits)
            
            all_labels.append(labels.cpu().numpy())
            all_preds.append(probs.cpu().numpy())

    all_labels = np.vstack(all_labels)
    all_preds = np.vstack(all_preds)

    try:
        # Calculate AUC for each class, then average
        auc_scores = roc_auc_score(all_labels, all_preds, average=None)
        mean_auc = np.mean(auc_scores)
        print(f"Mean AUC-ROC: {mean_auc:.4f}")
        return auc_scores, mean_auc
    except ValueError:
        print("Error in AUC calculation. Ensure all classes have both positive and negative samples in the validation set.")
        return None, None

## Step 7: Results & Comparison
Use this section to plot the validation AUC over epochs for: Baseline, ViT-Full, ViT-Partial, and ViT-LoRA.